# 08 – Baselines, confidence intervals and cross-validation

**Goal:** complete the benchmarking from the proposal:
1. Add the **Framingham Risk Score** – the tool doctors use today
2. Report every AUC with a **95% confidence interval**
3. Test whether differences between models are **statistically real**
4. Check stability with **5-fold cross-validation**

## Step 1 – Load data

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"; os.environ["OMP_NUM_THREADS"] = "1"
import warnings; warnings.filterwarnings("ignore")
from xgboost import XGBClassifier
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(42)

df = pd.read_csv("../data/framingham.csv")
df.loc[(df.currentSmoker == 0) & (df.cigsPerDay.isna()), "cigsPerDay"] = 0
df = df.drop(columns=["education"])
X = df.drop(columns=["TenYearCHD"]); y = df["TenYearCHD"]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)
print(len(X_train), len(X_cal), len(X_test))

2542 848 848


## Step 2 – Framingham Risk Score (office-based)

The standard Framingham score needs HDL cholesterol, which this dataset does not have. We use the **office-based (non-laboratory) version** (D'Agostino et al., 2008), which uses **BMI instead of cholesterol** – age, sex, BMI, systolic BP, BP treatment, smoking and diabetes. It is a fixed published formula: **no training**.

Note: it predicts general CVD; our outcome is CHD, so we compare ranking (AUC), not exact percentages.

In [2]:
def framingham_office_risk(data):
    """10-year general CVD risk, D'Agostino et al. 2008 (non-laboratory, BMI-based model)."""
    age = np.log(data["age"])
    bmi = np.log(data["BMI"].fillna(data["BMI"].median()))
    sbp = np.log(data["sysBP"])
    treated = data["BPMeds"].fillna(0) == 1
    smoker = data["currentSmoker"]
    diabetes = data["diabetes"]
    men = data["male"] == 1

    s_men = (3.11296 * age + 0.79277 * bmi + np.where(treated, 1.92672, 1.85508) * sbp
             + 0.70953 * smoker + 0.53160 * diabetes)
    s_women = (2.72107 * age + 0.51125 * bmi + np.where(treated, 2.88267, 2.81291) * sbp
               + 0.61868 * smoker + 0.77763 * diabetes)

    risk_men = 1 - 0.88431 ** np.exp(s_men - 23.9388)
    risk_women = 1 - 0.94833 ** np.exp(s_women - 26.0145)
    return np.where(men, risk_men, risk_women)

frs_test = framingham_office_risk(X_test)
print("FRS test AUC:", round(roc_auc_score(y_test, frs_test), 3))
print("Mean FRS risk:", round(frs_test.mean() * 100, 1), "% | actual CHD rate:", round(y_test.mean() * 100, 1), "%")

FRS test AUC: 0.702
Mean FRS risk: 12.3 % | actual CHD rate: 15.2 %


## Step 3 – Train the ML models

In [3]:
def build_lr():
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
                     ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))])

def build_xgb(y_tr):
    w = (y_tr == 0).sum() / (y_tr == 1).sum()
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("model", XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8,
                                             colsample_bytree=0.8, scale_pos_weight=w,
                                             random_state=42, n_jobs=1))])

lr = build_lr().fit(X_train, y_train)
xgb = build_xgb(y_train).fit(X_train, y_train)

preds = {"Framingham Risk Score (office)": frs_test,
         "Logistic Regression": lr.predict_proba(X_test)[:, 1],
         "XGBoost": xgb.predict_proba(X_test)[:, 1]}

## Step 4 – Test AUC with 95% confidence intervals (bootstrap)

Resample the 848 test patients 2,000 times and recompute AUC each time. The middle 95% of results is the confidence interval.

In [4]:
def bootstrap_ci(y_true, p, n=2000, seed=0):
    y_true = np.asarray(y_true); rng = np.random.default_rng(seed); aucs = []
    for _ in range(n):
        i = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) < 2: continue
        aucs.append(roc_auc_score(y_true[i], p[i]))
    return np.percentile(aucs, [2.5, 97.5])

rows = []
for name, p in preds.items():
    lo, hi = bootstrap_ci(y_test, p)
    rows.append({"model": name, "test_AUC": roc_auc_score(y_test, p), "95%_CI_low": lo, "95%_CI_high": hi})
pd.DataFrame(rows).round(3)

,model,test_AUC,95%_CI_low,95%_CI_high
0,Framingham Risk Score (office),0.702,0.653,0.747
1,Logistic Regression,0.696,0.648,0.741
2,XGBoost,0.679,0.627,0.728


## Step 5 – Are the differences real? (paired bootstrap)

If the 95% CI of the difference **includes 0**, the two models are not significantly different.

In [5]:
def paired_diff(y_true, p_a, p_b, n=2000, seed=0):
    y_true = np.asarray(y_true); rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) < 2: continue
        d.append(roc_auc_score(y_true[i], p_a[i]) - roc_auc_score(y_true[i], p_b[i]))
    return np.mean(d), np.percentile(d, [2.5, 97.5])

for a, b in [("Logistic Regression", "Framingham Risk Score (office)"),
             ("XGBoost", "Framingham Risk Score (office)"),
             ("XGBoost", "Logistic Regression")]:
    m, (lo, hi) = paired_diff(y_test, preds[a], preds[b])
    print(f"{a} vs {b}: {m:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

Logistic Regression vs Framingham Risk Score (office): -0.006 (95% CI -0.027 to +0.014)
XGBoost vs Framingham Risk Score (office): -0.023 (95% CI -0.052 to +0.006)
XGBoost vs Logistic Regression: -0.017 (95% CI -0.038 to +0.005)


## Step 6 – 5-fold cross-validation

Split all 4,238 patients into 5 parts; train on 4, test on 1, repeat 5 times. A small standard deviation means results are stable, not a lucky split.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {"Framingham Risk Score (office)": [], "Logistic Regression": [], "XGBoost": []}

for fold, (tr, te) in enumerate(cv.split(X, y), 1):
    Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
    cv_scores["Framingham Risk Score (office)"].append(roc_auc_score(yte, framingham_office_risk(Xte)))
    cv_scores["Logistic Regression"].append(roc_auc_score(yte, build_lr().fit(Xtr, ytr).predict_proba(Xte)[:, 1]))
    cv_scores["XGBoost"].append(roc_auc_score(yte, build_xgb(ytr).fit(Xtr, ytr).predict_proba(Xte)[:, 1]))

cv_table = pd.DataFrame(cv_scores, index=[f"fold {i}" for i in range(1, 6)])
cv_table.loc["mean"] = cv_table.mean()
cv_table.loc["std"] = cv_table.iloc[:5].std()
cv_table.round(3)

,Framingham Risk Score (office),Logistic Regression,XGBoost
fold 1,0.715,0.731,0.705
fold 2,0.716,0.702,0.693
fold 3,0.698,0.717,0.680
fold 4,0.713,0.723,0.707
fold 5,0.743,0.755,0.738
mean,0.717,0.726,0.705
std,0.016,0.019,0.022


## Summary

| Model | Test AUC (95% CI) | 5-fold CV AUC (mean ± SD) |
|---|---|---|
| Framingham Risk Score (office) | 0.702 (0.653–0.747) | 0.717 ± 0.016 |
| Logistic Regression | 0.696 (0.648–0.741) | 0.726 ± 0.019 |
| XGBoost | 0.677 (0.624–0.727) | 0.705 ± 0.022 |

- **No significant differences** between the three models (all paired 95% CIs include 0)
- Results are **stable** across folds (SD ≈ 0.02)
- The Framingham Risk Score performs well here because it was **derived from this same population** – its known weakness is in **non-Western populations** (e.g. South Asians), which is tested in the Sri Lankan validation stage
- FRS **under-predicts** risk in this cohort (mean 12.3% vs actual 15.2%), supporting the need for recalibration
- Framework value lies in **federation, uncertainty, explainability and population recalibration**, not raw AUC on Framingham data